# Industrial Pump Condition Classification

**AI Services MS2 | Supplementary Classification Practice | 60 marks | 20 tests**

A maintenance team must classify pump inspections as **Healthy**, **Inspect**, or **Urgent**. Complete the independent functions, compare two classifiers on held-back data, save the winning fitted pipeline and score queued inspections without retraining.

**Input files:** `pump_train.csv`, `pump_test.csv`, `pump_predict.csv`. Keep all three beside this notebook. The model/report outputs are generated only when the workflow cells are run.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

TARGET = "condition_label"
CLASS_NAMES = ["Healthy", "Inspect", "Urgent"]
raw_numeric_cols = [
    "inlet_pressure_bar", "outlet_pressure_bar", "flow_volume_m3",
    "runtime_hours", "motor_temperature_c", "ambient_temperature_c",
    "vibration_mm_s", "operating_load_pct"
]
category_cols = ["pump_type", "maintenance_zone", "shift_code"]
numeric_features = raw_numeric_cols + [
    "inspection_hour", "pressure_gain_bar", "flow_rate_m3h", "temperature_rise_c"
]
features = numeric_features + category_cols

## Function A1 - Common loader

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean one CSV and create pump features. See instruction paper."""
    # TODO: IDs, timestamps, numeric/category cleaning and 4 engineered features

    df = pd.read_csv(filename,dtype={'inspectiion_id':'string'})
    df['inspection_id'] = df['inspection_id'].str.strip().replace('',pd.NA)
    df = df.dropna(subset=['inspection_id'])
    df = df.drop_duplicates(subset=['inspection_id'],keep='first')

    df['inspected_at'] = pd.to_datetime(df['inspected_at'],format='mixed',errors='coerce',dayfirst=True)
    df = df.dropna(subset=['inspected_at'])

    df['inspection_hour'] = df['inspected_at'].dt.hour

    for col in raw_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in category_cols:
        df[col] = df[col].astype(str).str.strip().str.lower().replace('',pd.NA).fillna('unknown')

    df['shift_code'] = df['shift_code'].astype(str).replace(r'\.0+$','',regex=True)

    df['pressure_gain_bar'] = df['outlet_pressure_bar'] - df['inlet_pressure_bar']

    df['temperature_rise_c'] = df['motor_temperature_c'] - df['ambient_temperature_c']

    df['flow_rate_m3h'] = df['flow_volume_m3']/df['runtime_hours'].where(df['runtime_hours']>0)

    return df
    raise NotImplementedError

## Function A2 - Reusable classification pipeline

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Return unfitted preprocessing plus the passed classifier."""
    # TODO: ColumnTransformer with numeric and categorical branches

    num_pipeline = Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ('numeric',num_pipeline,numeric_features),
        ('category',OneHotEncoder(handle_unknown='ignore'),category_cols)
    ])

    return Pipeline([
        ('preprocessor',preprocessor),
        ('model',algorithm)
    ])
    raise NotImplementedError

## Function 1 - Prepare labeled datasets

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load files, normalize valid classes and return X_train, X_test, y_train, y_test."""
    # TODO: Filter targets and use one stable LabelEncoder mapping

    train_df = load_data(train_file)
    test_df = load_data(test_file)

    for df in (train_df,test_df):
        df[TARGET] = df[TARGET].astype(str).str.strip().str.title()

    train_df = train_df[train_df[TARGET].isin(CLASS_NAMES)]
    test_df = test_df[test_df[TARGET].isin(CLASS_NAMES)]

    encoder = LabelEncoder().fit(CLASS_NAMES)

    X_train = train_df[features]
    X_test = test_df[features]

    y_train = pd.Series(encoder.transform(train_df[TARGET]),index=train_df.index,name=TARGET)
    y_test = pd.Series(encoder.transform(test_df[TARGET]),index=test_df.index,name=TARGET)

    return X_train,X_test,y_train,y_test
    raise NotImplementedError

## Function 2 - Train candidate classifiers

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Fit Logistic Regression and Random Forest classifier pipelines."""
    # TODO: Compare the two prescribed classifiers

    models = {
        'logistic_regression' : LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        'random_forest' : RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)
    }

    trained_models = {}

    for name,algo in models.items():
        pipeline = build_pipeline(algo)
        pipeline.fit(X_train,y_train)
        trained_models[name] = pipeline

    return trained_models
    raise NotImplementedError

## Function 3 - Evaluate and compare models

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame,y_test: pd.Series) -> pd.DataFrame:
    """Return accuracy, macro F1 and weighted F1, sorted by macro F1."""
    # TODO: Rank without refitting

    results = []

    class_labels = np.arange(3)

    for name,pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({
            'model_name' : name,
            'accuracy' : accuracy_score(y_test,predictions),
            'macro_f1' : f1_score(y_test,predictions,average='macro',zero_division=0,labels=class_labels),
            'weighted_f1' : f1_score(y_test,predictions,average='weighted',zero_division=0,labels=class_labels)
        })

    comparsion = pd.DataFrame(results)

    return comparsion.sort_values(['macro_f1','accuracy','model_name'],ascending=[False,False,True]).reset_index(drop=True)
    raise NotImplementedError

## Function 4 - Select best model

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return first-ranked model name and the same fitted pipeline."""

    best_name = comparison.iloc[0]['model_name']
    best_model = trained_models[best_name]

    return best_name,best_model
    raise NotImplementedError

## Function 5 - Predict pump condition

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict_condition(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return labels and Urgent-class probabilities for cleaned queued records."""
    # TODO: inverse_transform and predict_proba
    raise NotImplementedError

## Training workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Persist model, comparison and confusion matrix when called."""
    raise NotImplementedError

## Prediction workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict() -> pd.DataFrame:
    """Reload saved pipeline and export queued predictions when called."""
    raise NotImplementedError

## Run the workflows

Run the **training** cell first. You may run **prediction** later in a fresh kernel after executing only the imports and definition cells, provided `best_model.pkl` is present. Do not fit inside the prediction workflow.

In [ ]:
train()

In [ ]:
predict()